In [94]:
# Setup and imports
from pathlib import Path
import sys, re
import numpy as np
import pandas as pd

def find_repo_root(name="masters_thesis_sdg"):
    p = Path.cwd().resolve()
    for x in [p, *p.parents]:
        if x.name == name:
            return x
    raise FileNotFoundError(f"Could not find repo root: {name}")

PROJECT_ROOT = find_repo_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "lai2023"
RESULTS_DIR = PROJECT_ROOT / "results" / "voting"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils_analysis.lai2023_loading import load_annotation_splits, build_game_lookup
from utils_analysis.llm_votes import load_llm_votes
from utils_analysis.row_builders import build_llm_binary_rows
from utils_analysis.strategy_features import rows_to_xy, get_feature_names
from utils_analysis.logreg_eval import run_tuned_logreg_with_classweight_search, metrics_table

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
DATA_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\lai2023
RESULTS_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting


In [ ]:
# CONFIG

llm = "unsloth_gemma-4-31B-it"
prompt_family = "v4_t0"

annot_splits = load_annotation_splits(DATA_DIR)
game_lookup, split_lookup = build_game_lookup(annot_splits)

print("Total games in annotation lookup:", len(game_lookup))

split_counts = {
    dataset: {split: len(games) for split, games in split_map.items()}
    for dataset, split_map in annot_splits.items()
}
display(pd.DataFrame(split_counts).T)

In [95]:
# IMPORTANT: normalise session names to be conssitent with annotations 
def canonical_session_key(x):
    x = str(x).strip()
    x = x.replace("#", " ")
    x = re.sub(r"\s+", " ", x)
    return x.lower()

def canonical_game_id(x):
    x = str(x).strip()
    m = re.search(r"\d+", x)
    return f"game{int(m.group())}" if m else x.lower()

def canonical_key(dataset, session_key, game_id):
    return (
        dataset,
        canonical_session_key(session_key),
        canonical_game_id(game_id),
    )

canonical_game_lookup = {}
canonical_split_lookup = {}

for key, game in game_lookup.items():
    dataset, session_key, game_id = key
    ckey = canonical_key(dataset, session_key, game_id)

    canonical_game_lookup[ckey] = game
    canonical_split_lookup[ckey] = split_lookup[key]

print("Canonical lookup size:", len(canonical_game_lookup))

Canonical lookup size: 191


In [ ]:
# find all runs for the given LLM and prompt 

def discover_runs(results_dir, llm, prompt_family):
    model_dir = results_dir / llm

    if not model_dir.exists():
        matches = [d for d in results_dir.iterdir() if d.is_dir() and llm in d.name]
        if len(matches) != 1:
            raise FileNotFoundError(f"Could not uniquely resolve LLM dir for: {llm}")
        model_dir = matches[0]
        print("Resolved model dir:", model_dir.name)

    pf = prompt_family.replace("prompt_", "")
    pat = re.compile(rf"^prompt_{re.escape(pf)}(?:_run_(\d+))?$")

    runs = []
    for d in model_dir.iterdir():
        m = pat.match(d.name)
        if d.is_dir() and m:
            runs.append({
                "run": int(m.group(1)) if m.group(1) else 0,
                "prompt_version": d.name,
            })

    runs = sorted(runs, key=lambda x: x["run"])

    if not runs:
        raise FileNotFoundError(f"No runs found for {prompt_family} in {model_dir}")

    return runs

prompt_runs = discover_runs(RESULTS_DIR, llm, prompt_family)
display(pd.DataFrame(prompt_runs))

In [97]:
# lr helpers

COMMON_C_GRID = (0.01, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.4, 2.0, 3.0)

COMMON_CLASS_WEIGHT_GRID = (
    None,
    "balanced",
    {0: 1, 1: 1.5},
    {0: 1, 1: 2},
    {0: 1, 1: 3},
    {0: 1, 1: 4},
    {0: 1, 1: 5},
)

def canonicalize_votes_df(votes_df):
    df = votes_df.copy()
    df["session_key"] = df["session_key"].apply(canonical_session_key)
    df["game_id"] = df["game_id"].apply(canonical_game_id)
    return df

def build_rows_for_run(run_info, feature_kind):
    votes_df, bad_df = load_llm_votes(
        RESULTS_DIR,
        llm=llm,
        prompt_version=run_info["prompt_version"],
    )

    votes_df = canonicalize_votes_df(votes_df)

    split_rows, unmatched_df = build_llm_binary_rows(
        votes_df,
        canonical_game_lookup,
        canonical_split_lookup,
        feature_kind=feature_kind,
    )

    if len(unmatched_df) > 0:
        print(f"WARNING: {len(unmatched_df)} unmatched games for {run_info['prompt_version']}")
        display(unmatched_df.head())

    return split_rows, votes_df, bad_df, unmatched_df

def train_one_run(split_rows, feature_kind):
    feature_names = get_feature_names(feature_kind)

    X_train, y_train = rows_to_xy(split_rows["train"])
    X_val, y_val = rows_to_xy(split_rows["val"])
    X_test, y_test = rows_to_xy(split_rows["test"])

    res = run_tuned_logreg_with_classweight_search(
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        feature_names,
        c_grid=COMMON_C_GRID,
        class_weight_grid=COMMON_CLASS_WEIGHT_GRID,
        solver="liblinear",
        optimize_for="f1",
    )

    metrics_df = metrics_table(res)[["split", "f1", "auc_prob"]].copy()
    coef_df = res["coef_df"].sort_values("coef", ascending=False).copy()

    return res, metrics_df, coef_df

def run_llm_candidate_experiment(feature_kind):
    metrics = []
    coefs = []
    results = []

    for run_info in prompt_runs:
        run = run_info["run"]
        prompt_version = run_info["prompt_version"]

        print(f"Running {feature_kind} | {prompt_version}")

        split_rows, votes_df, bad_df, unmatched_df = build_rows_for_run(
            run_info,
            feature_kind,
        )

        res, metrics_df, coef_df = train_one_run(split_rows, feature_kind)

        metrics_df["run"] = run
        metrics_df["prompt_version"] = prompt_version
        metrics_df["feature_kind"] = feature_kind

        coef_df["run"] = run
        coef_df["prompt_version"] = prompt_version
        coef_df["feature_kind"] = feature_kind

        metrics.append(metrics_df)
        coefs.append(coef_df)

        results.append({
            "run": run,
            "prompt_version": prompt_version,
            "result": res,
            "split_rows": split_rows,
            "votes_df": votes_df,
            "bad_df": bad_df,
            "unmatched_df": unmatched_df,
        })

    return {
        "feature_kind": feature_kind,
        "results": results,
        "metrics": pd.concat(metrics, ignore_index=True),
        "coefs": pd.concat(coefs, ignore_index=True),
    }

In [98]:
check_rows, check_votes, check_bad, check_unmatched = build_rows_for_run(
    prompt_runs[0],
    feature_kind="candidate_aggregate",
)

print("Votes:", len(check_votes))
print("Bad votes:", len(check_bad))
print("Unmatched:", len(check_unmatched))

for split in ["train", "val", "test"]:
    keys = {
        (r["dataset"], r["session_key"], r["game_id"])
        for r in check_rows[split]
    }

    X, y = rows_to_xy(check_rows[split])

    print()
    print(split)
    print("games:", len(keys))
    print("rows:", len(check_rows[split]))
    print("positives:", int(y.sum()))
    print("positive_rate:", y.mean())
    print("feature_sum_min:", X.sum(axis=1).min())
    print("feature_sum_max:", X.sum(axis=1).max())
    print("zero_feature_rows:", int((X.sum(axis=1) == 0).sum()))

Votes: 191
Bad votes: 0
Unmatched: 0

train
games: 133
rows: 595
positives: 133
positive_rate: 0.2235294117647059
feature_sum_min: 0.0
feature_sum_max: 1.0000001
zero_feature_rows: 9

val
games: 20
rows: 93
positives: 20
positive_rate: 0.21505376344086022
feature_sum_min: 0.9999999
feature_sum_max: 1.0000001
zero_feature_rows: 0

test
games: 38
rows: 176
positives: 38
positive_rate: 0.2159090909090909
feature_sum_min: 0.0
feature_sum_max: 1.0000001
zero_feature_rows: 10


In [99]:
#run aggregate candidate-only experiment
llm_cand = run_llm_candidate_experiment("candidate_aggregate")

llm_cand_metrics = llm_cand["metrics"]
llm_cand_coefs = llm_cand["coefs"]

print("Aggregate candidate-only metrics")
display(llm_cand_metrics)

print("Aggregate candidate-only coefficients")
display(llm_cand_coefs)

Running candidate_aggregate | prompt_v3_run_1
Aggregate candidate-only metrics


,split,f1,auc_prob,run,prompt_version,feature_kind
0,train,0.272374,0.567889,1,prompt_v3_run_1,candidate_aggregate
1,val,0.444444,0.589041,1,prompt_v3_run_1,candidate_aggregate
2,test,0.265060,0.543288,1,prompt_v3_run_1,candidate_aggregate


Aggregate candidate-only coefficients


,feature,coef,run,prompt_version,feature_kind
0,candidate_Identity Declaration,0.635486,1,prompt_v3_run_1,candidate_aggregate
1,candidate_Defense,0.171412,1,prompt_v3_run_1,candidate_aggregate
2,candidate_Call for Action,0.010441,1,prompt_v3_run_1,candidate_aggregate
3,candidate_Evidence,-0.094486,1,prompt_v3_run_1,candidate_aggregate
4,candidate_No Strategy,-0.312518,1,prompt_v3_run_1,candidate_aggregate
5,candidate_Interrogation,-0.598994,1,prompt_v3_run_1,candidate_aggregate
6,candidate_Accusation,-1.314149,1,prompt_v3_run_1,candidate_aggregate


In [100]:
llm_temp_cand = run_llm_candidate_experiment("candidate_temporal")

llm_temp_metrics = llm_temp_cand["metrics"]
llm_temp_coefs = llm_temp_cand["coefs"]

print("Temporal candidate-only metrics")
display(llm_temp_metrics)

print("Temporal candidate-only coefficients")
display(llm_temp_coefs)

Running candidate_temporal | prompt_v3_run_1
Temporal candidate-only metrics


,split,f1,auc_prob,run,prompt_version,feature_kind
0,train,0.337079,0.587825,1,prompt_v3_run_1,candidate_temporal
1,val,0.472727,0.627397,1,prompt_v3_run_1,candidate_temporal
2,test,0.336134,0.537757,1,prompt_v3_run_1,candidate_temporal


Temporal candidate-only coefficients


,feature,coef,run,prompt_version,feature_kind
0,candidate_late_Identity Declaration,1.341847,1,prompt_v3_run_1,candidate_temporal
1,candidate_early_Defense,0.602881,1,prompt_v3_run_1,candidate_temporal
2,candidate_early_Call for Action,0.392484,1,prompt_v3_run_1,candidate_temporal
3,candidate_early_Evidence,0.148292,1,prompt_v3_run_1,candidate_temporal
4,candidate_late_No Strategy,0.074792,1,prompt_v3_run_1,candidate_temporal
5,candidate_late_Call for Action,-0.012533,1,prompt_v3_run_1,candidate_temporal
6,candidate_late_Defense,-0.053924,1,prompt_v3_run_1,candidate_temporal
7,candidate_late_Evidence,-0.068612,1,prompt_v3_run_1,candidate_temporal
8,candidate_early_Interrogation,-0.119502,1,prompt_v3_run_1,candidate_temporal
9,candidate_early_Identity Declaration,-0.132111,1,prompt_v3_run_1,candidate_temporal


In [101]:
def summarize_test_metrics(metrics_df):
    return (
        metrics_df[metrics_df["split"].eq("test")]
        .groupby("feature_kind")
        .agg(
            mean_f1=("f1", "mean"),
            std_f1=("f1", "std"),
            mean_auc=("auc_prob", "mean"),
            std_auc=("auc_prob", "std"),
        )
        .reset_index()
    )

metric_summary = pd.concat(
    [
        summarize_test_metrics(llm_cand_metrics),
        summarize_test_metrics(llm_temp_metrics),
    ],
    ignore_index=True,
)

display(metric_summary)

,feature_kind,mean_f1,std_f1,mean_auc,std_auc
0,candidate_aggregate,0.265060,NaN,0.543288,NaN
1,candidate_temporal,0.336134,NaN,0.537757,NaN


In [102]:
def summarize_raw_coefs(coefs_df):
    return (
        coefs_df
        .groupby(["feature_kind", "feature"])
        .agg(
            mean_coef=("coef", "mean"),
            std_coef=("coef", "std"),
            min_coef=("coef", "min"),
            max_coef=("coef", "max"),
        )
        .reset_index()
        .sort_values("mean_coef", ascending=False)
    )

llm_cand_coef_summary = summarize_raw_coefs(llm_cand_coefs)
llm_temp_coef_summary = summarize_raw_coefs(llm_temp_coefs)

print("Aggregate raw coefficient summary")
display(llm_cand_coef_summary)

print("Temporal raw coefficient summary")
display(llm_temp_coef_summary)

Aggregate raw coefficient summary


,feature_kind,feature,mean_coef,std_coef,min_coef,max_coef
4,candidate_aggregate,candidate_Identity Declaration,0.635486,NaN,0.635486,0.635486
2,candidate_aggregate,candidate_Defense,0.171412,NaN,0.171412,0.171412
1,candidate_aggregate,candidate_Call for Action,0.010441,NaN,0.010441,0.010441
3,candidate_aggregate,candidate_Evidence,-0.094486,NaN,-0.094486,-0.094486
6,candidate_aggregate,candidate_No Strategy,-0.312518,NaN,-0.312518,-0.312518
5,candidate_aggregate,candidate_Interrogation,-0.598994,NaN,-0.598994,-0.598994
0,candidate_aggregate,candidate_Accusation,-1.314149,NaN,-1.314149,-1.314149


Temporal raw coefficient summary


,feature_kind,feature,mean_coef,std_coef,min_coef,max_coef
11,candidate_temporal,candidate_late_Identity Declaration,1.341847,NaN,1.341847,1.341847
2,candidate_temporal,candidate_early_Defense,0.602881,NaN,0.602881,0.602881
1,candidate_temporal,candidate_early_Call for Action,0.392484,NaN,0.392484,0.392484
3,candidate_temporal,candidate_early_Evidence,0.148292,NaN,0.148292,0.148292
13,candidate_temporal,candidate_late_No Strategy,0.074792,NaN,0.074792,0.074792
8,candidate_temporal,candidate_late_Call for Action,-0.012533,NaN,-0.012533,-0.012533
9,candidate_temporal,candidate_late_Defense,-0.053924,NaN,-0.053924,-0.053924
10,candidate_temporal,candidate_late_Evidence,-0.068612,NaN,-0.068612,-0.068612
5,candidate_temporal,candidate_early_Interrogation,-0.119502,NaN,-0.119502,-0.119502
4,candidate_temporal,candidate_early_Identity Declaration,-0.132111,NaN,-0.132111,-0.132111


In [103]:
OUT_DIR = PROJECT_ROOT / "results" / "analysis" / "llm_candidate_only_logreg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

llm_cand_metrics.to_csv(OUT_DIR / f"{llm}_{prompt_family}_candidate_aggregate_metrics.csv", index=False)
llm_temp_metrics.to_csv(OUT_DIR / f"{llm}_{prompt_family}_candidate_temporal_metrics.csv", index=False)

llm_cand_coefs.to_csv(OUT_DIR / f"{llm}_{prompt_family}_candidate_aggregate_raw_coefs.csv", index=False)
llm_temp_coefs.to_csv(OUT_DIR / f"{llm}_{prompt_family}_candidate_temporal_raw_coefs.csv", index=False)

metric_summary.to_csv(OUT_DIR / f"{llm}_{prompt_family}_test_metric_summary.csv", index=False)

print("Saved to:", OUT_DIR)

Saved to: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\analysis\llm_candidate_only_logreg
